# Fruit Segmentation — Google Colab Training

Trains **ConvNeXt-V2 U-Net** or **Swin-V2 U-Net** on the Fresh & Rotten Fruits dataset.

**Runtime**: Runtime → Change runtime type → **T4 GPU**

---
### Steps
1. Mount Google Drive
2. Clone repo & install dependencies
3. Upload / link dataset
4. Choose model config
5. Train
6. Visualise results

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ── 2. Clone repo and install ────────────────────────────────────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/fruit-segmentation.git"  # update this
REPO_DIR = "/content/fruit-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Install uv then sync dependencies
!pip install uv -q
!uv pip install -e . --system -q

print("✓ Dependencies installed")

In [ ]:
# ── 3. Dataset setup ─────────────────────────────────────────────────────
# Option A: download from Kaggle
# !pip install kaggle -q
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !kaggle datasets download -d <dataset-slug> -p data/raw --unzip

# Option B: copy preprocessed data from Drive
DRIVE_DATA = "/content/drive/MyDrive/fruit_segmentation_data"
LOCAL_DATA = "/content/fruit-segmentation/data/processed"

import os, shutil

if os.path.exists(DRIVE_DATA):
    if not os.path.exists(LOCAL_DATA):
        shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f"✓ Dataset at {LOCAL_DATA}")
else:
    print("⚠ Dataset not found — update DRIVE_DATA path or use Option A above")

In [ ]:
# ── 4. Choose model and size ─────────────────────────────────────────────
MODEL = "convnext_unet"  # "convnext_unet" or "swin_unet"
SIZE = "tiny"  # "tiny", "small", or "base"

CONFIG = f"config/{MODEL}.yaml"
print(f"Config: {CONFIG}  |  Size: {SIZE}")

In [ ]:
# ── 5. Verify GPU ────────────────────────────────────────────────────────
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 6. Quick model sanity check ──────────────────────────────────────────
import sys

sys.path.insert(0, "/content/fruit-segmentation")

from config.config_loader import load_config
from models import build_model

cfg = load_config(CONFIG)
cfg["model"]["size"] = SIZE

model = build_model(cfg)
counts = model.count_parameters()
print(f"Model    : {cfg['model']['name']}_{SIZE}")
print(f"Backbone : {counts['backbone']/1e6:.1f}M params")
print(f"Decoder  : {counts['decoder']/1e6:.1f}M params")
print(f"Total    : {counts['total']/1e6:.1f}M params")

# Forward pass smoke test
x = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = model(x)
print(f"Output shape: {out.shape}  ✓")

In [ ]:
# ── 7. Train ─────────────────────────────────────────────────────────────
!python train/train.py \
    --config {CONFIG} \
    --size {SIZE} \
    --device cuda

In [ ]:
# ── 8. Visualise training curves ─────────────────────────────────────────
import glob
from visualization.plot_metrics import plot_training_curves, compare_logs_from_dir

# Find latest log for this model
pattern = f"logs/{MODEL}_{SIZE}-*.csv"
logs = sorted(glob.glob(pattern))
if logs:
    fig_path = plot_training_curves(logs[-1], show=True)
    print(f"Saved: {fig_path}")
else:
    print(f"No log files found matching {pattern}")

In [ ]:
# ── 9. Compare all trained models ────────────────────────────────────────
compare_logs_from_dir("logs/", metric="miou")

In [ ]:
# ── 10. Run inference on a sample image ──────────────────────────────────
import glob, os

CKPT = f"checkpoints/best/{MODEL}_{SIZE}_best.pth"
SAMPLE_IMAGE = "data/processed/images/test/"  # directory or single file
OUTPUT_DIR = "predictions/"

if os.path.exists(CKPT):
    !python inference/inference.py \
        --config {CONFIG} \
        --checkpoint {CKPT} \
        --input {SAMPLE_IMAGE} \
        --output {OUTPUT_DIR} \
        --overlay
else:
    print(f"Checkpoint not found: {CKPT}")

In [ ]:
# ── 11. Display a prediction overlay ─────────────────────────────────────
import glob
from PIL import Image
import matplotlib.pyplot as plt

overlays = sorted(glob.glob(f"{OUTPUT_DIR}/*_overlay.png"))
if overlays:
    fig, axes = plt.subplots(1, min(3, len(overlays)), figsize=(15, 5))
    if len(overlays) == 1:
        axes = [axes]
    for ax, path in zip(axes, overlays[:3]):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).stem, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No overlay images found — run the inference cell first.")

In [ ]:
# ── 12. Save results back to Drive ───────────────────────────────────────
import shutil, datetime

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
DRIVE_SAVE = f"/content/drive/MyDrive/fruit_seg_results/{MODEL}_{SIZE}_{TIMESTAMP}"

os.makedirs(DRIVE_SAVE, exist_ok=True)

# Copy checkpoints and logs
for src in ["checkpoints", "logs"]:
    if os.path.exists(src):
        shutil.copytree(src, f"{DRIVE_SAVE}/{src}", dirs_exist_ok=True)

print(f"✓ Results saved to {DRIVE_SAVE}")